# Objective Quest DATAQUEST 5.0 — V11 Historical Prediction Portfolio Ordinal Stacker

CPU-only meta-learning over **completed historical OOF/test prediction artifacts**. No base-model retraining, no GPU, no V9/V10 partial artifacts.

Frozen champion: CPU recovery ensemble (Public LB **0.48334**). Headline validation is unique-respondent, 5-fold cross-fitted, and submission is generated **only if the preregistered promotion gate passes**.

In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES']=''
os.environ.setdefault('OMP_NUM_THREADS','4')
os.environ.setdefault('MKL_NUM_THREADS','4')
os.environ.setdefault('OPENBLAS_NUM_THREADS','4')
os.environ.setdefault('NUMEXPR_NUM_THREADS','4')

import sys, json, math, random, hashlib, zipfile, platform, warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy
from scipy.optimize import minimize
from scipy.stats import spearmanr
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import cohen_kappa_score, roc_auc_score, log_loss, mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')
SEED=42
random.seed(SEED); np.random.seed(SEED)

@dataclass(frozen=True)
class Config:
    target:str='life_satisfaction'
    id_col:str='id'
    folds:int=5
    c_grid:tuple=(0.01,0.1,1.0,10.0)
    bootstrap_reps:int=2000
    expected_train_rows:int=95609
    expected_train_ids:int=94663
    expected_test_rows:int=23902
    expected_test_ids:int=23666
    recovery_lb:float=0.48334
    falsify_qwk_gain:float=0.0005
    falsify_auc12_gain:float=0.001
    promote_qwk_gain:float=0.0015
    promote_boot_p:float=0.90
    promote_min_fold_delta:float=-0.0015
    challenge_qwk_gain:float=0.0025
    challenge_auc12_gain:float=0.004
CFG=Config()
OUT=Path('/kaggle/working/oqdq5_v11') if Path('/kaggle/working').exists() else Path('./oqdq5_v11')
OUT.mkdir(parents=True,exist_ok=True)
print('V11 output:',OUT)
print('CPU-only:',os.environ.get('CUDA_VISIBLE_DEVICES')=='')
print('numpy',np.__version__,'pandas',pd.__version__,'scipy',scipy.__version__,'sklearn',sklearn.__version__)

V11 output: /kaggle/working/oqdq5_v11
CPU-only: True
numpy 2.0.2 pandas 2.3.3 scipy 1.16.3 sklearn 1.6.1


## 1. Competition data and historical artifact discovery

The notebook accepts either an extracted historical `results/` tree or a ZIP containing `results/v2/oof_v2.csv`. It hard-fails on missing/ambiguous sources.

In [2]:
def locate_competition_dir():
    roots=[Path('/kaggle/input'),Path('/mnt/data')]
    cwd=Path.cwd().resolve()
    if cwd != Path('/') and not str(cwd).startswith('/proc'):
        roots.append(cwd)
    req={'train_final.csv','test_final.csv'}
    found=[]
    for root in roots:
        if not root.exists(): continue
        for p in [root]+[x for x in root.rglob('*') if x.is_dir()]:
            if all((p/f).exists() for f in req): found.append(p)
    uniq=[]
    for p in found:
        rp=p.resolve()
        if rp not in uniq: uniq.append(rp)
    if not uniq: raise FileNotFoundError('train_final.csv/test_final.csv not found.')
    # Prefer a directory also containing sample_submission.csv.
    uniq.sort(key=lambda p:(not (p/'sample_submission.csv').exists(),len(str(p))))
    return uniq[0]

class ArtifactStore:
    def __init__(self, root=None, zip_path=None):
        self.root=Path(root) if root else None
        self.zip_path=Path(zip_path) if zip_path else None
        self.zf=zipfile.ZipFile(self.zip_path) if self.zip_path else None
    def exists(self, rel):
        rel=str(rel).replace('\\','/')
        if self.root: return (self.root/rel).exists()
        return rel in self.zf.namelist()
    def read_csv(self, rel, **kw):
        rel=str(rel).replace('\\','/')
        if self.root: return pd.read_csv(self.root/rel,**kw)
        with self.zf.open(rel) as f: return pd.read_csv(f,**kw)
    def source(self): return str(self.root if self.root else self.zip_path)
    def close(self):
        if self.zf: self.zf.close()

def locate_artifact_store():
    roots=[Path('/kaggle/input'),Path('/mnt/data')]
    cwd=Path.cwd().resolve()
    if cwd != Path('/') and not str(cwd).startswith('/proc'):
        roots.append(cwd)
    dir_hits=[]; zip_hits=[]
    marker=Path('results/v2/oof_v2.csv')
    for root in roots:
        if not root.exists(): continue
        # Extracted tree: either root/results/... or root itself is results/ parent.
        for p in root.rglob('oof_v2.csv'):
            if p.as_posix().endswith('/results/v2/oof_v2.csv'):
                base=p.parents[2]
                if (base/marker).exists(): dir_hits.append(base)
        for z in root.rglob('*.zip'):
            try:
                with zipfile.ZipFile(z) as zf:
                    if marker.as_posix() in zf.namelist(): zip_hits.append(z)
            except zipfile.BadZipFile:
                pass
    # De-duplicate and prefer extracted source; otherwise require unique matching ZIP.
    dir_hits=list(dict.fromkeys(map(lambda p:p.resolve(),dir_hits)))
    zip_hits=list(dict.fromkeys(map(lambda p:p.resolve(),zip_hits)))
    if dir_hits:
        dir_hits.sort(key=lambda p:len(str(p)))
        return ArtifactStore(root=dir_hits[0])
    if len(zip_hits)==1: return ArtifactStore(zip_path=zip_hits[0])
    if not zip_hits: raise FileNotFoundError('Historical results archive/tree not found. Upload the completed results.zip as a Kaggle input.')
    raise RuntimeError(f'Ambiguous historical ZIPs containing the required results tree: {zip_hits}')

DATA_DIR=locate_competition_dir(); STORE=locate_artifact_store()
train_raw=pd.read_csv(DATA_DIR/'train_final.csv',usecols=['id','life_satisfaction'])
test_raw=pd.read_csv(DATA_DIR/'test_final.csv',usecols=['id'])
assert len(train_raw)==CFG.expected_train_rows and train_raw.id.nunique()==CFG.expected_train_ids
assert len(test_raw)==CFG.expected_test_rows and test_raw.id.nunique()==CFG.expected_test_ids
assert set(train_raw.life_satisfaction.unique())=={1,2,3,4}
print('DATA_DIR:',DATA_DIR)
print('ARTIFACT_SOURCE:',STORE.source())

DATA_DIR: /kaggle/input/competitions/oq-dq-5
ARTIFACT_SOURCE: /kaggle/input/datasets/riantamaputra/dataquest-2026-historical-results


## 2. Explicit artifact eligibility and provenance

Only completed V1–V8.x score streams with explicit OOF↔test column mappings are eligible. V9 and V10 are intentionally absent.

In [3]:
# name: (version, oof_file, test_file, oof_col, test_col, family)
STREAM_SPECS={
# core tree family
'v1_lgb':('v1','oof_v1.csv','test_predictions_v1.csv','oof_lgb_continuous','test_lgb_continuous','core_tree'),
'v1_xgb':('v1','oof_v1.csv','test_predictions_v1.csv','oof_xgb_continuous','test_xgb_continuous','core_tree'),
'v1_cat':('v1','oof_v1.csv','test_predictions_v1.csv','oof_cat_continuous','test_cat_continuous','core_tree'),
'v2_lgb_core':('v2','oof_v2.csv','test_predictions_v2.csv','oof_lgb_core_continuous','test_lgb_core_continuous','core_tree'),
'v2_xgb_core':('v2','oof_v2.csv','test_predictions_v2.csv','oof_xgb_core_continuous','test_xgb_core_continuous','core_tree'),
'v2_cat_core':('v2','oof_v2.csv','test_predictions_v2.csv','oof_cat_core_continuous','test_cat_core_continuous','core_tree'),
'v5_lgb':('v5_1','oof_v5.csv','test_predictions_v5.csv','oof_lgb','pred_lgb','core_tree'),
'v5_xgb':('v5_1','oof_v5.csv','test_predictions_v5.csv','oof_xgb','pred_xgb','core_tree'),
'v5_cat':('v5_1','oof_v5.csv','test_predictions_v5.csv','oof_cat','pred_cat','core_tree'),
'v6_lgb':('v6','oof_v6.csv','test_predictions_v6.csv','oof_lgb','pred_lgb','core_tree'),
'v6_xgb':('v6','oof_v6.csv','test_predictions_v6.csv','oof_xgb','pred_xgb','core_tree'),
'v6_cat':('v6','oof_v6.csv','test_predictions_v6.csv','oof_cat','pred_cat','core_tree'),
'v61_lgb_core':('v6_1','oof_v6.csv','test_predictions_v6.csv','oof_lgb_core','test_lgb_core','core_tree'),
'v61_xgb_core':('v6_1','oof_v6.csv','test_predictions_v6.csv','oof_xgb_core','test_xgb_core','core_tree'),
'v61_cat_core':('v6_1','oof_v6.csv','test_predictions_v6.csv','oof_cat_core','test_cat_core','core_tree'),
# categorical / ordinal
'v2_lgb_cat':('v2','oof_v2.csv','test_predictions_v2.csv','oof_lgb_categorical_continuous','test_lgb_categorical_continuous','categorical_ordinal'),
'v2_xgb_cat':('v2','oof_v2.csv','test_predictions_v2.csv','oof_xgb_categorical_continuous','test_xgb_categorical_continuous','categorical_ordinal'),
'v2_lgb_mono':('v2','oof_v2.csv','test_predictions_v2.csv','oof_lgb_monotonic_continuous','test_lgb_monotonic_continuous','categorical_ordinal'),
'v2_lgb_mc':('v2','oof_v2.csv','test_predictions_v2.csv','oof_lgb_multiclass_expected_continuous','test_lgb_multiclass_expected_continuous','categorical_ordinal'),
'v2_lgb_cum':('v2','oof_v2.csv','test_predictions_v2.csv','oof_lgb_cumulative_ordinal_continuous','test_lgb_cumulative_ordinal_continuous','categorical_ordinal'),
'v61_xgb_cat':('v6_1','oof_v6.csv','test_predictions_v6.csv','oof_xgb_categorical','test_xgb_categorical','categorical_ordinal'),
'v61_lgb_cum':('v6_1','oof_v6.csv','test_predictions_v6.csv','oof_lgb_cumulative_ordinal','test_lgb_cumulative_ordinal','categorical_ordinal'),
# structural
'v3_lgb_struct':('v3','oof_v3.csv','test_predictions_v3.csv','lgb_struct','lgb_struct','structural'),
'v3_xgb_struct':('v3','oof_v3.csv','test_predictions_v3.csv','xgb_struct','xgb_struct','structural'),
'v3_cat_dual':('v3','oof_v3.csv','test_predictions_v3.csv','cat_dual','cat_dual','structural'),
# explicit all-categorical representation
'v61_allcat':('v6_1','oof_v6.csv','test_predictions_v6.csv','oof_catboost_allcat','test_catboost_allcat','allcat'),
# alternate completed final/blend scores
'v1_ensemble':('v1','oof_v1.csv','test_predictions_v1.csv','oof_ensemble_continuous','test_ensemble_continuous','alternate_gbdt'),
'v2_core':('v2','oof_v2.csv','test_predictions_v2.csv','oof_core_continuous','test_core_continuous','alternate_gbdt'),
'v2_final':('v2','oof_v2.csv','test_predictions_v2.csv','oof_final_continuous','test_final_continuous','alternate_gbdt'),
'v3_final':('v3','oof_v3.csv','test_predictions_v3.csv','final_score','final_score','alternate_gbdt'),
'v5_final':('v5_1','oof_v5.csv','test_predictions_v5.csv','oof_final_honest_continuous','pred_final_continuous','alternate_gbdt'),
'v6_final':('v6','oof_v6.csv','test_predictions_v6.csv','oof_final_crossfit','pred_final_continuous','alternate_gbdt'),
'v61_final':('v6_1','oof_v6.csv','test_predictions_v6.csv','oof_final_crossfit','pred_final_continuous','alternate_gbdt'),
'v7_stack':('v7','oof_v7.csv','test_scores_v7.csv','v7_stack','v7_stack','alternate_gbdt'),
'v8_stack':('v8','oof_v8.csv','test_scores_v8.csv','v8_stack','v8_stack','alternate_gbdt'),
'v81_final':('v8_1','oof_v8_1.csv','test_scores_v8_1.csv','final_score','final_score','alternate_gbdt'),
'v82_final':('v8_2','oof_v8_2.csv','test_scores_v8_2.csv','final_score','final_score','alternate_gbdt'),
'v83_final':('v8_3','oof_v8_3.csv','test_scores_v8_3.csv','final_score','final_score','alternate_gbdt'),
# repeated exact/near-exact V2 anchors
'v7_anchor':('v7','oof_v7.csv','test_scores_v7.csv','v2_anchor','v2_anchor','repeated_anchor'),
'v8_anchor':('v8','oof_v8.csv','test_scores_v8.csv','v2_anchor','v2_anchor','repeated_anchor'),
'v81_anchor':('v8_1','oof_v8_1.csv','test_scores_v8_1.csv','v2_exact_score','v2_exact_score','repeated_anchor'),
'v82_anchor':('v8_2','oof_v8_2.csv','test_scores_v8_2.csv','v2_exact_score','v2_exact_score','repeated_anchor'),
'v83_anchor':('v8_3','oof_v8_3.csv','test_scores_v8_3.csv','v2_exact_score','v2_exact_score','repeated_anchor'),
}
FAMILIES=('core_tree','categorical_ordinal','structural','alternate_gbdt','allcat','repeated_anchor')
assert not any(k.startswith(('v9','v10')) for k in STREAM_SPECS)
print('Explicit eligible streams:',len(STREAM_SPECS))

Explicit eligible streams: 43


In [4]:
OOF_META={
'v1':('fold','life_satisfaction'),'v2':('primary_fold','life_satisfaction'),'v3':('fold','target'),
'v5_1':('fold','life_satisfaction'),'v6':('fold','life_satisfaction'),'v6_1':('primary_fold','life_satisfaction'),
'v7':('primary_fold','life_satisfaction'),'v8':('primary_fold','life_satisfaction'),
'v8_1':('primary_fold','life_satisfaction'),'v8_2':('primary_fold','life_satisfaction'),'v8_3':('primary_fold','life_satisfaction')}
OOF_FILES={'v1':'oof_v1.csv','v2':'oof_v2.csv','v3':'oof_v3.csv','v5_1':'oof_v5.csv','v6':'oof_v6.csv','v6_1':'oof_v6.csv','v7':'oof_v7.csv','v8':'oof_v8.csv','v8_1':'oof_v8_1.csv','v8_2':'oof_v8_2.csv','v8_3':'oof_v8_3.csv'}
TEST_FILES={'v1':'test_predictions_v1.csv','v2':'test_predictions_v2.csv','v3':'test_predictions_v3.csv','v5_1':'test_predictions_v5.csv','v6':'test_predictions_v6.csv','v6_1':'test_predictions_v6.csv','v7':'test_scores_v7.csv','v8':'test_scores_v8.csv','v8_1':'test_scores_v8_1.csv','v8_2':'test_scores_v8_2.csv','v8_3':'test_scores_v8_3.csv'}

# Canonical entity/fold order is frozen V2 OOF and first-appearance unique test id order.
v2o=STORE.read_csv('results/v2/oof_v2.csv',usecols=['id','life_satisfaction','primary_fold'])
assert v2o.groupby('id').life_satisfaction.nunique().max()==1
assert v2o.groupby('id').primary_fold.nunique().max()==1
canon=v2o.drop_duplicates('id',keep='first').reset_index(drop=True)
assert len(canon)==CFG.expected_train_ids and set(canon.primary_fold.unique())==set(range(5))
train_entity=train_raw.drop_duplicates('id',keep='first')
chk=canon[['id','life_satisfaction']].merge(train_entity,on='id',suffixes=('_artifact','_data'),validate='one_to_one')
assert np.array_equal(chk.life_satisfaction_artifact.to_numpy(),chk.life_satisfaction_data.to_numpy())
train_ids=canon.id.to_numpy(); y=canon.life_satisfaction.astype(int).to_numpy(); folds=canon.primary_fold.astype(int).to_numpy()
test_entity=test_raw.drop_duplicates('id',keep='first').reset_index(drop=True); test_ids=test_entity.id.to_numpy()

# Cache each CSV once.
oof_cache={}; test_cache={}
for v in OOF_FILES:
    oof_cache[v]=STORE.read_csv(f'results/{v}/{OOF_FILES[v]}')
    test_cache[v]=STORE.read_csv(f'results/{v}/{TEST_FILES[v]}')

def infer_test_id(v,df):
    if 'id' in df: return df['id']
    if 'test_feature_id' in df: return df['test_feature_id']
    if 'test_id' in df: return df['test_id']
    if 'row_index' in df:
        ri=df['row_index'].to_numpy()
        assert np.array_equal(ri,np.arange(len(test_raw)))
        return test_raw['id'].reset_index(drop=True)
    raise KeyError(f'No valid test-id mapping for {v}')

def collapse_oof(v,pred_col):
    df=oof_cache[v]; fold_col,target_col=OOF_META[v]
    for c in ['id',target_col,fold_col,pred_col]:
        if c not in df: raise KeyError(f'{v} missing {c}')
    g=df[['id',target_col,fold_col,pred_col]].groupby('id',sort=False)
    if g[target_col].nunique().max()!=1 or g[fold_col].nunique().max()!=1: raise AssertionError(f'{v}: duplicate id target/fold inconsistency')
    span=(g[pred_col].max()-g[pred_col].min()).max()
    if not np.isfinite(span) or span>1e-8: raise AssertionError(f'{v}:{pred_col} duplicate predictions differ (max span {span})')
    z=g.agg({target_col:'first',fold_col:'first',pred_col:'mean'}).reset_index()
    z=canon[['id']].merge(z,on='id',how='left',validate='one_to_one')
    if z.isna().any().any(): raise AssertionError(f'{v}:{pred_col} incomplete OOF alignment')
    if not np.array_equal(z[target_col].astype(int).to_numpy(),y): raise AssertionError(f'{v}:{pred_col} target mismatch')
    fold_match=float(np.mean(z[fold_col].astype(int).to_numpy()==folds))
    return z[pred_col].astype(float).to_numpy(),fold_match

def collapse_test(v,pred_col):
    df=test_cache[v].copy()
    if pred_col not in df: raise KeyError(f'{v} missing test {pred_col}')
    df['_entity_id']=infer_test_id(v,df).to_numpy()
    g=df[['_entity_id',pred_col]].groupby('_entity_id',sort=False)
    span=(g[pred_col].max()-g[pred_col].min()).max()
    if not np.isfinite(span) or span>1e-8: raise AssertionError(f'{v}:{pred_col} duplicate test predictions differ (max span {span})')
    z=g[pred_col].mean().reset_index().rename(columns={'_entity_id':'id'})
    z=test_entity[['id']].merge(z,on='id',how='left',validate='one_to_one')
    if z[pred_col].isna().any(): raise AssertionError(f'{v}:{pred_col} incomplete test alignment')
    return z[pred_col].astype(float).to_numpy()

streams_oof={}; streams_test={}; prov=[]
for name,(v,of,tf,oc,tc,fam) in STREAM_SPECS.items():
    assert of==OOF_FILES[v] and tf==TEST_FILES[v]
    a,fold_match=collapse_oof(v,oc); b=collapse_test(v,tc)
    if not (np.isfinite(a).all() and np.isfinite(b).all()): raise AssertionError(f'nonfinite stream {name}')
    streams_oof[name]=a; streams_test[name]=b
    prov.append({'stream':name,'version':v,'family':fam,'oof_col':oc,'test_col':tc,'fold_match_to_canonical':fold_match,'oof_n':len(a),'test_n':len(b)})
prov_df=pd.DataFrame(prov)
prov_df.to_csv(OUT/'stream_provenance.csv',index=False)
print(prov_df.groupby('family').size())
print('Noncanonical historical OOF folds:',prov_df.loc[prov_df.fold_match_to_canonical<0.999,['version','fold_match_to_canonical']].drop_duplicates().to_dict('records'))

family
allcat                  1
alternate_gbdt         12
categorical_ordinal     7
core_tree              15
repeated_anchor         5
structural              3
dtype: int64
Noncanonical historical OOF folds: [{'version': 'v5_1', 'fold_match_to_canonical': 0.19910630341316038}, {'version': 'v6', 'fold_match_to_canonical': 0.19910630341316038}]


## 3. De-duplicate identical score streams, aggregate predefined families, reconstruct frozen recovery champion

In [5]:
# Identical OOF+test streams are counted once globally.
kept=[]; dup_rows=[]
for name in STREAM_SPECS:
    duplicate_of=None
    for k in kept:
        if np.array_equal(streams_oof[name],streams_oof[k]) and np.array_equal(streams_test[name],streams_test[k]):
            duplicate_of=k; break
    if duplicate_of is None: kept.append(name)
    else: dup_rows.append({'stream':name,'duplicate_of':duplicate_of})
pd.DataFrame(dup_rows,columns=['stream','duplicate_of']).to_csv(OUT/'duplicate_streams.csv',index=False)

family_members={}
family_oof={}; family_test={}
for fam in FAMILIES:
    members=[n for n in STREAM_SPECS if STREAM_SPECS[n][-1]==fam and n in kept]
    if not members: raise RuntimeError(f'Family {fam} has no unique eligible streams')
    family_members[fam]=members
    family_oof[fam]=np.mean(np.column_stack([streams_oof[n] for n in members]),axis=1)
    family_test[fam]=np.mean(np.column_stack([streams_test[n] for n in members]),axis=1)

REC_NAMES=('v2_final','v5_final','v61_final','v7_stack')
REC_FOLD_WEIGHTS={
0:(0.6131481145225551,0.1984633896432871,0.18838849583415773,0.0),
1:(0.5736087705637004,0.2130793852900037,0.20117728057551384,0.012134563570782105),
2:(0.5260926035320435,0.22651646669428574,0.20045779567769734,0.04693313409597358),
3:(0.412997730762594,0.12147998345945334,0.2995554634343377,0.16596682234361498),
4:(0.4419367210113302,0.27031416820745807,0.2416404622741877,0.04610864850702414)}
REC_FOLD_THRESH={
0:(1.5694444444444446,2.011875,2.4941666666666666),1:(1.57,2.011875,2.5100000000000002),
2:(1.57,2.0162500000000003,2.5100000000000002),3:(1.5683333333333334,2.011875,2.524166666666667),
4:(1.568888888888889,2.014375,2.5100000000000002)}
REC_FINAL_W=np.array((0.5333147962860486,0.2160045364556684,0.20393903979487613,0.0467416274634069))
REC_FINAL_T=np.array((1.5694444444444446,2.011875,2.5100000000000002))
REC_EXPECTED_QWK=0.48186721909562624

rec_X=np.column_stack([streams_oof[n] for n in REC_NAMES]); rec_T=np.column_stack([streams_test[n] for n in REC_NAMES])
recovery_oof=np.empty(len(y)); recovery_class=np.empty(len(y),dtype=int)
for f in range(5):
    m=folds==f; recovery_oof[m]=rec_X[m]@np.array(REC_FOLD_WEIGHTS[f]); t=np.array(REC_FOLD_THRESH[f])
    recovery_class[m]=1+(recovery_oof[m]>=t[0])+(recovery_oof[m]>=t[1])+(recovery_oof[m]>=t[2])
recovery_test=rec_T@REC_FINAL_W
recovery_test_class=1+(recovery_test>=REC_FINAL_T[0])+(recovery_test>=REC_FINAL_T[1])+(recovery_test>=REC_FINAL_T[2])
rec_qwk=cohen_kappa_score(y,recovery_class,weights='quadratic')
assert abs(rec_qwk-REC_EXPECTED_QWK)<1e-12, (rec_qwk,REC_EXPECTED_QWK)

with open(OUT/'family_members.json','w') as f: json.dump(family_members,f,indent=2)
print('Unique streams:',len(kept),'duplicates removed:',len(dup_rows))
print('Frozen recovery OOF QWK:',rec_qwk,'Public LB:',CFG.recovery_lb)

Unique streams: 40 duplicates removed: 3
Frozen recovery OOF QWK: 0.48186721909562624 Public LB: 0.48334


## 4. Leakage-safe meta-feature construction and ordinal utilities

Threshold-dependent disagreement features always receive thresholds estimated from the relevant meta-training partition only.

In [6]:
def qwk(y_true,y_pred): return float(cohen_kappa_score(y_true,y_pred,weights='quadratic'))
def discretize(s,t):
    t=np.asarray(t,float); return (1+(s>=t[0])+(s>=t[1])+(s>=t[2])).astype(int)

def fit_qwk_thresholds(y_true,score,start=(1.57,2.01,2.51)):
    y_true=np.asarray(y_true,int); score=np.asarray(score,float)
    def obj(t):
        t=np.sort(np.asarray(t,float))
        if t[1]-t[0]<0.02 or t[2]-t[1]<0.02: return 10.0
        return -qwk(y_true,discretize(score,t))
    bounds=((1.15,1.95),(1.55,2.45),(2.05,3.25))
    res=minimize(obj,np.array(start,float),method='Powell',bounds=bounds,options={'maxiter':250,'xtol':1e-5,'ftol':1e-8})
    t=np.sort(res.x)
    if not (t[0]<t[1]<t[2]): raise RuntimeError('Threshold optimizer returned invalid ordering')
    return t

def pava3(P):
    # Exact three-point non-increasing isotonic projection (equal weights).
    P=np.asarray(P,float); out=np.empty_like(P)
    for i,(a,b,c) in enumerate(P):
        vals=[a,b,c]; blocks=[[0,a,1],[1,b,1],[2,c,1]]
        j=0
        while j < len(blocks)-1:
            if blocks[j][1] < blocks[j+1][1]:
                w=blocks[j][2]+blocks[j+1][2]
                av=(blocks[j][1]*blocks[j][2]+blocks[j+1][1]*blocks[j+1][2])/w
                blocks[j:j+2]=[[blocks[j][0],av,w]]
                j=max(0,j-1)
            else: j+=1
        pos=0
        for start,av,w in blocks:
            out[i,pos:pos+w]=av; pos+=w
    return np.clip(out,1e-7,1-1e-7)

fam_names=list(FAMILIES)
F_oof=np.column_stack([family_oof[f] for f in fam_names]); F_test=np.column_stack([family_test[f] for f in fam_names])

def make_meta_features(F,rec,thresholds):
    F=np.asarray(F,float); rec=np.asarray(rec,float); t=np.asarray(thresholds,float)
    data={}
    for j,n in enumerate(fam_names): data[f'family__{n}']=F[:,j]
    data['recovery_score']=rec
    data['family_mean']=F.mean(1); data['family_std']=F.std(1); data['family_min']=F.min(1); data['family_max']=F.max(1); data['family_range']=F.max(1)-F.min(1)
    for j,n in enumerate(fam_names): data[f'recovery_minus__{n}']=rec-F[:,j]
    for k,tk in enumerate(t,1): data[f'prop_family_above_t{k}']=(F>=tk).mean(1)
    data['recovery_margin_t1']=rec-t[0]; data['recovery_abs_margin_t1']=np.abs(rec-t[0])
    X=pd.DataFrame(data)
    if not np.isfinite(X.to_numpy()).all(): raise AssertionError('nonfinite meta features')
    return X

def fit_heads(X,y,C):
    heads=[]
    for k in (1,2,3):
        z=(y>k).astype(int)
        pipe=Pipeline([('scale',StandardScaler()),('lr',LogisticRegression(C=float(C),solver='lbfgs',max_iter=2000,random_state=SEED))])
        pipe.fit(X,z); heads.append(pipe)
    return heads

def predict_heads(heads,X):
    P=np.column_stack([h.predict_proba(X)[:,1] for h in heads])
    P=pava3(P); score=1.0+P.sum(1)
    return P,score

def cumulative_losses(y,P):
    losses=[]
    for j,k in enumerate((1,2,3)): losses.append(log_loss((y>k).astype(int),P[:,j],labels=[0,1]))
    score=1+P.sum(1)
    return float(np.mean(losses)),float(np.sqrt(mean_squared_error(y,score)))

def boundary_auc(y,s,cut): return float(roc_auc_score((y>cut).astype(int),s))
def auc_1v2(y,s):
    m=np.isin(y,[1,2]); return float(roc_auc_score((y[m]==2).astype(int),s[m]))

## 5. Nested meta-model selection and frozen 5-fold outer evaluation

For each outer fold, `C` is selected by leave-one-historical-fold-out validation inside the other four folds. Primary selector: mean cumulative binary log loss; tie-break: expected-score RMSE.

In [7]:
inner_rows=[]; selected=[]
oof_P=np.full((len(y),3),np.nan); v11_oof=np.full(len(y),np.nan)

for outer in range(5):
    tr=folds!=outer; va=folds==outer
    inner_folds=[f for f in range(5) if f!=outer]
    # Threshold-derived features do not depend on C; build each inner split exactly once.
    split_cache={}
    for iv in inner_folds:
        itr=tr & (folds!=iv); iva=(folds==iv)
        thr_i=fit_qwk_thresholds(y[itr],recovery_oof[itr])
        split_cache[iv]=(itr,iva,make_meta_features(F_oof[itr],recovery_oof[itr],thr_i),make_meta_features(F_oof[iva],recovery_oof[iva],thr_i))
    c_stats=[]
    for C in CFG.c_grid:
        lls=[]; rms=[]
        for iv in inner_folds:
            itr,iva,Xtr_i,Xva_i=split_cache[iv]
            heads=fit_heads(Xtr_i,y[itr],C); P,_=predict_heads(heads,Xva_i)
            ll,rm=cumulative_losses(y[iva],P); lls.append(ll); rms.append(rm)
        row={'outer_fold':outer,'C':C,'mean_logloss':float(np.mean(lls)),'mean_rmse':float(np.mean(rms))}
        inner_rows.append(row); c_stats.append(row)
    c_stats=sorted(c_stats,key=lambda r:(round(r['mean_logloss'],12),round(r['mean_rmse'],12),r['C']))
    bestC=float(c_stats[0]['C']); selected.append({'outer_fold':outer,'C':bestC})
    thr=fit_qwk_thresholds(y[tr],recovery_oof[tr])
    Xtr=make_meta_features(F_oof[tr],recovery_oof[tr],thr); Xva=make_meta_features(F_oof[va],recovery_oof[va],thr)
    heads=fit_heads(Xtr,y[tr],bestC); P,score=predict_heads(heads,Xva)
    oof_P[va]=P; v11_oof[va]=score
    print('outer',outer,'C',bestC,'n',int(va.sum()))

assert np.isfinite(v11_oof).all() and np.isfinite(oof_P).all()
inner_df=pd.DataFrame(inner_rows); selected_df=pd.DataFrame(selected)
inner_df.to_csv(OUT/'inner_cv_selection.csv',index=False); selected_df.to_csv(OUT/'outer_selected_C.csv',index=False)

outer 0 C 0.01 n 18933
outer 1 C 0.01 n 18933
outer 2 C 0.01 n 18933
outer 3 C 0.01 n 18931
outer 4 C 0.01 n 18933


## 6. Cross-fitted QWK calibration, diagnostics, uncertainty, and gates

In [8]:
v11_class=np.empty(len(y),dtype=int); thr_rows=[]
for f in range(5):
    tr=folds!=f; va=folds==f
    t=fit_qwk_thresholds(y[tr],v11_oof[tr]); v11_class[va]=discretize(v11_oof[va],t)
    thr_rows.append({'fold':f,'t1':t[0],'t2':t[1],'t3':t[2]})
thr_df=pd.DataFrame(thr_rows); thr_df.to_csv(OUT/'crossfit_thresholds_v11.csv',index=False)

V11_QWK=qwk(y,v11_class); REC_QWK=qwk(y,recovery_class); DELTA_QWK=V11_QWK-REC_QWK
A12_V11=auc_1v2(y,v11_oof); A12_REC=auc_1v2(y,recovery_oof); DELTA_A12=A12_V11-A12_REC
AY1_V11=boundary_auc(y,v11_oof,1); AY1_REC=boundary_auc(y,recovery_oof,1)

fold_rows=[]
for f in range(5):
    m=folds==f
    qv=qwk(y[m],v11_class[m]); qr=qwk(y[m],recovery_class[m])
    fold_rows.append({'fold':f,'n':int(m.sum()),'v11_qwk':qv,'recovery_qwk':qr,'delta_qwk':qv-qr,
                      'v11_auc_1v2':auc_1v2(y[m],v11_oof[m]),'recovery_auc_1v2':auc_1v2(y[m],recovery_oof[m]),
                      'v11_rmse':float(np.sqrt(mean_squared_error(y[m],v11_oof[m]))),'recovery_rmse':float(np.sqrt(mean_squared_error(y[m],recovery_oof[m])))})
fold_df=pd.DataFrame(fold_rows); fold_df.to_csv(OUT/'fold_metrics_v11.csv',index=False)

# Global-threshold score is diagnostic only and never used for promotion/submission decisions.
global_t=fit_qwk_thresholds(y,v11_oof); global_qwk=qwk(y,discretize(v11_oof,global_t)); global_delta=global_qwk-REC_QWK

rng=np.random.default_rng(SEED+1100); boot=np.empty(CFG.bootstrap_reps)
N=len(y)
for b in range(CFG.bootstrap_reps):
    idx=rng.integers(0,N,N); boot[b]=qwk(y[idx],v11_class[idx])-qwk(y[idx],recovery_class[idx])
boot_lo,boot_hi=np.quantile(boot,[.025,.975]); boot_p=float(np.mean(boot>0))
pd.DataFrame({'delta_qwk':boot}).to_csv(OUT/'paired_bootstrap_v11.csv',index=False)

positive_folds=int((fold_df.delta_qwk>=0).sum()); min_fold=float(fold_df.delta_qwk.min())
resid_corr=float(np.corrcoef(v11_oof-y,recovery_oof-y)[0,1])
change=pd.crosstab(pd.Series(recovery_class,name='recovery'),pd.Series(v11_class,name='v11'),dropna=False)
change.to_csv(OUT/'label_change_matrix_v11.csv')
change_rate=float(np.mean(v11_class!=recovery_class))
extreme_prob_rate=float(np.mean((oof_P<=1.00001e-7)|(oof_P>=1-1.00001e-7)))
collapse=bool((np.std(v11_oof)<0.10) or extreme_prob_rate>0.20)
global_only=bool(DELTA_QWK<CFG.falsify_qwk_gain and global_delta>=CFG.falsify_qwk_gain)

falsified=bool(((DELTA_QWK<CFG.falsify_qwk_gain) and (DELTA_A12<CFG.falsify_auc12_gain)) or positive_folds<3 or global_only or collapse)
promoted=bool((not falsified) and DELTA_QWK>=CFG.promote_qwk_gain and positive_folds>=4 and min_fold>=CFG.promote_min_fold_delta and boot_p>=CFG.promote_boot_p and DELTA_A12>=0)
challenge=bool(promoted and (DELTA_QWK>=CFG.challenge_qwk_gain or DELTA_A12>=CFG.challenge_auc12_gain) and positive_folds>=4)

metrics={
'recovery_qwk':REC_QWK,'v11_crossfit_qwk':V11_QWK,'delta_qwk':DELTA_QWK,
'recovery_auc_1v2':A12_REC,'v11_auc_1v2':A12_V11,'delta_auc_1v2':DELTA_A12,
'recovery_auc_ygt1':AY1_REC,'v11_auc_ygt1':AY1_V11,'delta_auc_ygt1':AY1_V11-AY1_REC,
'v11_rmse':float(np.sqrt(mean_squared_error(y,v11_oof))),'recovery_rmse':float(np.sqrt(mean_squared_error(y,recovery_oof))),
'v11_mae':float(mean_absolute_error(y,v11_oof)),'recovery_mae':float(mean_absolute_error(y,recovery_oof)),
'v11_spearman':float(spearmanr(y,v11_oof)[0]),'recovery_spearman':float(spearmanr(y,recovery_oof)[0]),
'prediction_corr_vs_recovery':float(np.corrcoef(v11_oof,recovery_oof)[0,1]),'residual_corr_vs_recovery':resid_corr,'label_change_rate_vs_recovery':change_rate,'label_change_n_vs_recovery':int(np.sum(v11_class!=recovery_class)),
'bootstrap_p_gt_0':boot_p,'bootstrap_95':[float(boot_lo),float(boot_hi)],'positive_folds':positive_folds,'min_fold_delta':min_fold,
'global_qwk_diagnostic':global_qwk,'global_delta_diagnostic':global_delta,'global_only_red_flag':global_only,'collapse_red_flag':collapse,
'falsified':falsified,'promoted':promoted,'top3_challenge_gate':challenge}
print(json.dumps(metrics,indent=2))

{
  "recovery_qwk": 0.48186721909562624,
  "v11_crossfit_qwk": 0.4786620927851717,
  "delta_qwk": -0.0032051263104545535,
  "recovery_auc_1v2": 0.7657964184802133,
  "v11_auc_1v2": 0.7657748372584973,
  "delta_auc_1v2": -2.1581221716027166e-05,
  "recovery_auc_ygt1": 0.779980058800323,
  "v11_auc_ygt1": 0.7799566045925316,
  "delta_auc_ygt1": -2.3454207791373527e-05,
  "v11_rmse": 0.5155023915523831,
  "recovery_rmse": 0.5152669061211503,
  "v11_mae": 0.4252168545101659,
  "recovery_mae": 0.4268770243838855,
  "v11_spearman": 0.5100083011658126,
  "recovery_spearman": 0.5100481059960007,
  "prediction_corr_vs_recovery": 0.997157976099462,
  "residual_corr_vs_recovery": 0.9988344995270648,
  "label_change_rate_vs_recovery": 0.0429206765050759,
  "label_change_n_vs_recovery": 4063,
  "bootstrap_p_gt_0": 0.0,
  "bootstrap_95": [
    -0.004889257171583519,
    -0.0014616745706173496
  ],
  "positive_folds": 0,
  "min_fold_delta": -0.0062762517930975426,
  "global_qwk_diagnostic": 0.4815591

In [9]:
oof_out=pd.DataFrame({'id':train_ids,'life_satisfaction':y,'fold':folds,'recovery_score':recovery_oof,'recovery_class':recovery_class,
                      'v11_p_gt1':oof_P[:,0],'v11_p_gt2':oof_P[:,1],'v11_p_gt3':oof_P[:,2],'v11_score':v11_oof,'v11_class':v11_class})
oof_out.to_csv(OUT/'oof_v11.csv',index=False)

# Feature manifest records the exact columns used; threshold values themselves are split-local.
probe_t=np.array((1.57,2.01,2.51))
feature_names=make_meta_features(F_oof[:2],recovery_oof[:2],probe_t).columns.tolist()
with open(OUT/'meta_feature_manifest.json','w') as f:
    json.dump({'families':family_members,'feature_names':feature_names,'threshold_features_are_split_local':True,'excluded_versions':['v9','v10']},f,indent=2)

## 7. Gated final fit and submission

No test meta-model inference and no submission are produced unless the preregistered **promotion gate** passes.

In [10]:
submission_path=None; test_pred_path=None; final_C=None; final_thresholds=None
if promoted:
    # Aggregate frozen-CV selection: lowest mean inner log-loss across outer folds; RMSE tie-break.
    agg=(inner_df.groupby('C',as_index=False)[['mean_logloss','mean_rmse']].mean()
         .sort_values(['mean_logloss','mean_rmse','C']))
    final_C=float(agg.iloc[0].C)
    feat_t=fit_qwk_thresholds(y,recovery_oof)
    Xall=make_meta_features(F_oof,recovery_oof,feat_t); Xt=make_meta_features(F_test,recovery_test,feat_t)
    heads=fit_heads(Xall,y,final_C); test_P,test_score=predict_heads(heads,Xt)
    final_thresholds=thr_df[['t1','t2','t3']].median(axis=0).to_numpy()
    test_class=discretize(test_score,final_thresholds)
    if not (np.isfinite(test_score).all() and set(np.unique(test_class)).issubset({1,2,3,4})): raise AssertionError('Invalid test predictions')
    test_out=pd.DataFrame({'id':test_ids,'v11_p_gt1':test_P[:,0],'v11_p_gt2':test_P[:,1],'v11_p_gt3':test_P[:,2],
                           'v11_score':test_score,'v11_class':test_class,'recovery_score':recovery_test,'recovery_class':recovery_test_class})
    assert len(test_out)==CFG.expected_test_ids and test_out.id.nunique()==CFG.expected_test_ids
    test_pred_path=OUT/'test_predictions_v11.csv'; test_out.to_csv(test_pred_path,index=False)
    submission=pd.DataFrame({'id':test_ids,'life_satisfaction':test_class.astype(int)})
    assert submission.id.nunique()==len(submission)==CFG.expected_test_ids
    submission_path=OUT/'submission_v11.csv'; submission.to_csv(submission_path,index=False)
    print('PROMOTED: submission written ->',submission_path)
else:
    print('NOT PROMOTED: no V11 test inference and no submission file were produced.')

NOT PROMOTED: no V11 test inference and no submission file were produced.


## 8. Final provenance and summary

In [11]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''): h.update(chunk)
    return h.hexdigest()

summary={
'version':'v11_historical_prediction_portfolio_ordinal_stacker','seed':SEED,'cpu_only':True,'base_models_retrained':False,
'artifact_source':STORE.source(),'data_dir':str(DATA_DIR),'eligible_versions':sorted(set(v[0] for v in STREAM_SPECS.values())),
'excluded_versions':['v9','v10'],'n_explicit_streams':len(STREAM_SPECS),'n_unique_streams':len(kept),'families':family_members,
'frozen_recovery_public_lb':CFG.recovery_lb,'metrics':metrics,'selected_C_by_outer_fold':selected,
'final_C':final_C,'final_thresholds':None if final_thresholds is None else [float(x) for x in final_thresholds],
'submission_path':None if submission_path is None else str(submission_path),
'test_predictions_path':None if test_pred_path is None else str(test_pred_path),
'validation_note':'Meta-level nested validation uses completed row-wise OOF base predictions; base models are not refit inside V11 outer folds. Historical fold compatibility is recorded in stream_provenance.csv.',
'gates':{'falsification':{'delta_qwk_and_auc_floor':[CFG.falsify_qwk_gain,CFG.falsify_auc12_gain],'min_positive_folds':3},
         'promotion':{'delta_qwk':CFG.promote_qwk_gain,'positive_folds':4,'min_fold_delta':CFG.promote_min_fold_delta,'bootstrap_p':CFG.promote_boot_p,'auc_1v2_nonregression':True},
         'top3_challenge':{'delta_qwk':CFG.challenge_qwk_gain,'or_delta_auc_1v2':CFG.challenge_auc12_gain}},
'package_versions':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'scipy':scipy.__version__,'sklearn':sklearn.__version__}}
if submission_path: summary['submission_sha256']=sha256_file(submission_path)
with open(OUT/'v11_summary.json','w') as f: json.dump(summary,f,indent=2)
with open(OUT/'provenance_manifest.json','w') as f:
    json.dump({'artifact_source':STORE.source(),'data_dir':str(DATA_DIR),'stream_specs':STREAM_SPECS,'recovery_components':REC_NAMES,
               'recovery_final_weights':REC_FINAL_W.tolist(),'recovery_final_thresholds':REC_FINAL_T.tolist(),'seed':SEED},f,indent=2)
STORE.close()
print('Status:', 'PROMOTED' if promoted else 'FALSIFIED/NOT PROMOTED')
print('Summary:',OUT/'v11_summary.json')

Status: FALSIFIED/NOT PROMOTED
Summary: /kaggle/working/oqdq5_v11/v11_summary.json
